In [21]:
!pip install openai -q

In [26]:
import json
from openai import OpenAI

# Paste your OpenRouter API key
API_KEY = "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"

# IMPORTANT:
# base_url MUST be OpenRouter
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=API_KEY,
)

messages = [
    "My payment got deducted but service is not activated",
    "App crashes every time I login",
    "How to change my email address?"
]

results = []

for message in messages:

    prompt = f"""
You are a support ticket classifier.

Your task is to classify customer messages into:

Categories:
1. Billing
2. Technical Issue
3. Account
4. General Inquiry

Priority Levels:
- High
- Medium
- Low

Classification Rules:

Billing:
- payment issues
- refund issues
- deducted amount
- subscription charges
- invoice problems

Technical Issue:
- app crashes
- bugs
- loading issues
- login failures
- software malfunction

Account:
- email change
- password reset
- account settings
- profile issues

General Inquiry:
- informational questions
- pricing questions
- feature questions

Priority Rules:

High:
- payment deducted
- service unavailable
- app crash
- blocking issue

Medium:
- slow performance
- partial inconvenience

Low:
- informational requests
- simple account changes

Return ONLY valid JSON.

Example:
{{
  "category": "Billing",
  "priority": "High"
}}

Message:
"{message}"
"""

    try:

        response = client.chat.completions.create(
            model="openai/gpt-3.5-turbo",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        content = response.choices[0].message.content

        parsed = json.loads(content)

        results.append({
            "message": message,
            "category": parsed["category"],
            "priority": parsed["priority"]
        })

    except Exception as e:

        results.append({
            "message": message,
            "error": str(e)
        })

print(json.dumps(results, indent=4))

[
    {
        "message": "My payment got deducted but service is not activated",
        "category": "Billing",
        "priority": "High"
    },
    {
        "message": "App crashes every time I login",
        "category": "Technical Issue",
        "priority": "High"
    },
    {
        "message": "How to change my email address?",
        "category": "Account",
        "priority": "Low"
    }
]
